# Select Texas2k models and run the original numerical adapter

Use this notebook to compare real saved-case inference for all six final representatives and request one bounded portability probe per family using the original Phase I optimizer. It requires the model environment, acquired weights, and the **exact checksum-bound electrical input pack**. Public weather availability is separate from the exact electrical input contract.

This notebook was executed in a clean checkout. Its exact-input numerical modes report **unavailable**, never passed inference/OPS, while the recorded result analysis executes below. After acquiring the exact inputs, successful numerical runs are CPU portability examples; they do not replace the recorded production GPU searches or exact AC audits.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display, JSON, Image

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'artifacts/model_catalog.json').is_file(), 'Run from this repository or its notebooks directory.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ROOT / 'outputs' / ('notebook_' + RUN_ID)
OUT.mkdir(parents=True)


In [2]:
from genco_ops.catalog import models
from genco_ops.workflow import run_workflow
MODEL_IDS = [f'texas2k.{family}.{regime}' for family in ('genco_opf', 'opengridsfm') for regime in ('G500', 'G100_to_R400', 'G100_to_W400')]
# Set to an authorized external graph-pack directory, or use the acquired repository-local pack.
INFERENCE_INPUTS = None
OPS_INPUTS = ROOT / 'artifacts/private_inputs/texas2k_ops'
PROBE_MODELS = ['texas2k.genco_opf.G500', 'texas2k.opengridsfm.G500']
RUN_BOUNDED_PROBES = True
display(pd.DataFrame([{key: row[key] for key in ('model_id', 'role', 'weights_ready', 'saved_input_ready', 'ops_status')} for row in models(ROOT, 'texas2k')]))

,model_id,role,weights_ready,saved_input_ready,ops_status
0,texas2k.genco_opf.G100,intermediate,True,False,NOT_A_FINAL_REPRESENTATIVE
1,texas2k.genco_opf.G100_to_R400,final,True,False,COMPLETE
2,texas2k.genco_opf.G100_to_W400,final,True,False,COMPLETE
3,texas2k.genco_opf.G500,final,True,False,COMPLETE
4,texas2k.genco_opf.base,base,True,False,HISTORICAL_BASE_RESULT_SEPARATELY_IDENTIFIED
5,texas2k.opengridsfm.G100,intermediate,True,False,NOT_A_FINAL_REPRESENTATIVE
6,texas2k.opengridsfm.G100_to_R400,final,True,False,PENDING
7,texas2k.opengridsfm.G100_to_W400,final,True,False,PENDING
8,texas2k.opengridsfm.G500,final,True,False,PENDING
9,texas2k.opengridsfm.base,base,True,False,HISTORICAL_BASE_RESULT_SEPARATELY_IDENTIFIED


In [3]:
inference_results = []
for model_id in MODEL_IDS:
    destination = OUT / ('inference_' + model_id)
    plan = run_workflow(ROOT, 'texas2k', model_id, destination, mode='infer', inputs=INFERENCE_INPUTS)
    if plan['status'] == 'MISSING_ARTIFACTS':
        result = {'model_id': model_id, 'status': 'UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_PASSED_DEMO', 'missing_assets': plan['missing_assets']}
    else:
        result = run_workflow(ROOT, 'texas2k', model_id, destination, mode='infer', inputs=INFERENCE_INPUTS, execute=True)
    inference_results.append(result)
    display(JSON({'kind': 'saved_case_inference', **result}))
display(pd.DataFrame(inference_results)[['model_id', 'status']])

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

,model_id,status
0,texas2k.genco_opf.G500,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...
1,texas2k.genco_opf.G100_to_R400,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...
2,texas2k.genco_opf.G100_to_W400,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...
3,texas2k.opengridsfm.G500,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...
4,texas2k.opengridsfm.G100_to_R400,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...
5,texas2k.opengridsfm.G100_to_W400,UNAVAILABLE_EXACT_INPUT_OR_MODEL_ASSETS_NOT_A_...


## Bounded original-contract search

Each requested probe uses a fixed two-line topology `(117, 281)`, five selected load-service controls, and at most 20 actual model calls on CPU. It uses the same original continuous-search contract as Phase I. The probe is not a full topology search and does not perform AC optimization. GENCO and OpenGridSFM load their own native input/output adapters.

For a full GPU search, use the documented `run --mode published` CLI with `--execute`, the prepared input pack, and one CUDA GPU; it retains the 301-topology / 6,020-call maximum. No scheduler jobs are submitted by this notebook.

In [4]:
probe_results = []
if RUN_BOUNDED_PROBES:
    for model_id in PROBE_MODELS:
        if not (OPS_INPUTS / 'input_contract.json').is_file():
            result = {'status': 'UNAVAILABLE_EXACT_OPS_INPUT_NOT_A_PASSED_DEMO', 'model_id': model_id}
        else:
            result = run_workflow(ROOT, 'texas2k', model_id, OUT / ('probe_' + model_id), mode='published', inputs=OPS_INPUTS, device='cpu', probe_topology=[117, 281], execute=True)
            result['model_id'] = model_id
            if 'result' in result:
                result['result'] = str(Path(result['result']).relative_to(ROOT))
        probe_results.append(result)
        display(JSON({'kind': 'original_ops_probe', **result}))
else:
    display(JSON({'kind': 'original_ops_probe', 'status': 'NOT_REQUESTED_NO_NUMERICAL_DEMO_CLAIM'}))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## Recorded production results and validation

The GENCO results below are retained production search/AC outcomes. The six saved-case predictions above do not imply six completed production searches: the accepted native OpenGridSFM full OPS/AC results remain pending in this evidence cut.

Eight fits are certified. Per-case validation metrics are locally retained for seven checkpoints, excluding native G500. G means intact-grid, R means random-outage, and O retains its original stored outage-validation label; no new test population is implied. Residuals below report power-balance mismatches in physical units, not normalized training losses.

In [5]:
from genco_ops.results import recompute_retained_decision, texas_validation
display(pd.DataFrame([recompute_retained_decision(ROOT, regime) for regime in ('G500', 'G100_to_R400', 'G100_to_W400')]).round(8))
display(texas_validation(ROOT).round(4))
print('Fresh inference and bounded probes above are separate from retained production results. New AC solves: 0.')

,model,offline_branch_ids,native_r,a_r,shed,native_j,a_j,fresh_inference,fresh_ops,fresh_ac_audit
0,GENCO G500,"[117, 1548]",1.270111,1.164144,0.002697,1.016628,0.931855,False,False,False
1,GENCO G100 -> R400,"[117, 281]",1.450011,1.089042,0.000486,1.160106,0.871331,False,False,False
2,GENCO G100 -> W400,"[117, 281]",1.511185,1.089060,0.000546,1.209057,0.871357,False,False,False


,checkpoint_task,family,distribution,records,mean_absolute_p_mw,mean_absolute_q_mvar,mean_worst_bus_p_mw,mean_worst_bus_q_mvar,metric_source
0,v070.genco_opf.G100.s0.validation,genco_opf,G,100,84.5710,68.0406,4576.6096,7609.4697,GENCO physical evaluator
1,v070.genco_opf.G100.s0.validation,genco_opf,O,100,88.3867,68.8603,4653.6255,7609.4680,GENCO physical evaluator
2,v070.genco_opf.G100.s0.validation,genco_opf,R,100,85.4669,68.2800,4660.7580,7609.0689,GENCO physical evaluator
3,v070.genco_opf.G100_to_R400.s0.validation,genco_opf,G,100,50.6709,16.6038,9311.8529,3470.5470,GENCO physical evaluator
4,v070.genco_opf.G100_to_R400.s0.validation,genco_opf,O,100,55.7988,17.3199,9011.4442,3194.9986,GENCO physical evaluator
5,v070.genco_opf.G100_to_R400.s0.validation,genco_opf,R,100,51.5584,16.8230,9300.2767,3465.0076,GENCO physical evaluator
6,v070.genco_opf.G100_to_W400.s0.validation,genco_opf,G,100,51.9724,16.7047,9841.2927,3737.9700,GENCO physical evaluator
7,v070.genco_opf.G100_to_W400.s0.validation,genco_opf,O,100,52.3238,16.8122,9594.0497,3606.1526,GENCO physical evaluator
8,v070.genco_opf.G100_to_W400.s0.validation,genco_opf,R,100,52.7313,16.9416,9846.7643,3738.9368,GENCO physical evaluator
9,v070.genco_opf.G500.s0.validation,genco_opf,G,100,51.5638,20.5946,6264.7606,2086.6933,GENCO physical evaluator


Fresh inference and bounded probes above are separate from retained production results. New AC solves: 0.
